In [2]:
"""
vulnerability.py
HydroTerra - Flood vulnerability scoring
Owner: Pranaya

Combines water accumulation, low elevation, and settlement presence
into a single 0-1 risk score per grid cell.

Framing for viva/professor:
"We are not claiming real-world flood prediction accuracy. The score
is a relative simulation-based vulnerability indicator combining
water accumulation, low elevation, and settlement presence."
"""

import numpy as np
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_PATH = "/content/drive/MyDrive/HydroTerra"
SRC_PATH = f"{PROJECT_PATH}/src"

# Check that the folder really exists
print("Project exists:", os.path.exists(PROJECT_PATH))
print("src exists:", os.path.exists(SRC_PATH))

# Show all files inside src
print("\nFiles inside src:")
print(os.listdir(SRC_PATH))

# Add src to Python's import path
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("\nsrc added to Python path:", SRC_PATH)


def normalize(arr):
    """Rescale an array to [0, 1]."""
    mn = arr.min()
    mx = arr.max()
    return (arr - mn) / (mx - mn + 1e-8)


def calculate_risk(terrain, water, settlement,
                    w_water=0.55, w_elevation=0.30, w_settlement=0.15):
    """
    risk = 0.55 * water_amount + 0.30 * (1 - elevation) + 0.15 * settlement

    - More water  -> higher risk
    - Lower elevation -> higher risk
    - Being a settlement cell -> extra risk weight
    """
    normalized_height = normalize(terrain)
    low_elevation = 1 - normalized_height
    normalized_water = normalize(water)

    risk = (
        w_water * normalized_water
        + w_elevation * low_elevation
        + w_settlement * settlement
    )

    return np.clip(risk, 0, 1)


def make_settlement(size=128, y_range=(50, 65), x_range=(50, 70)):
    """Simple rectangular settlement footprint for testing."""
    settlement = np.zeros((size, size))
    settlement[y_range[0]:y_range[1], x_range[0]:x_range[1]] = 1
    return settlement


if __name__ == "__main__":
    from terrain import generate_terrain

    terrain = generate_terrain()
    water = np.random.random((128, 128)) * 0.1
    settlement = make_settlement()

    risk = calculate_risk(terrain, water, settlement)
    print("Risk shape:", risk.shape)
    print("Risk range:", risk.min(), risk.max())


Mounted at /content/drive
Project exists: True
src exists: True

Files inside src:
['vulnerability.py', 'simulation.py', 'terrain.py', 'terrain.ipynb', 'vulnerability.ipynb', 'simulation.ipynb']

src added to Python path: /content/drive/MyDrive/HydroTerra/src
Risk shape: (128, 128)
Risk range: 0.007423574358569269 0.8838557526268216
